# Testing BoARIO with GLORIA IO data


## Processing the GLORIA data
These cells parse the gloria dataset using pymrio, and set up the ARIOsiModel with test parameters, simulating a shock to Russian Agriculture (because eora does not disaggregate more). 

In [3]:
# import pymrio
# gloria_storage = "data/tmp/gloria57"

# Download one year
# gloria_log_v057_2009 = pymrio.download_gloria(gloria_storage, year=2009, version=57)

# Download all years
# gloria_log = pymrio.download_gloria(storage_folder=gloria_storage)

There is no information on how to parse gloria on pymrio (something to contribute), so it has to be added manually. Below are the steps to do so, including removing the supply matrices from the SUTs. 

Does not need rebalancing as the supply tables are in the offdiagonal. 

In [4]:
import pandas as pd

_sectors = ["Growing wheat",
"Growing maize",
"Growing cereals n.e.c",
"Growing leguminous crops and oil seeds",
"Growing rice", 
"Growing vegetables, roots, tubers",
"Growing sugar beet and cane",
"Growing tobacco",
"Growing fibre crops",
"Growing crops n.e.c.",
"Growing grapes",
"Growing fruits and nuts",
"Growing beverage crops (coffee, tea etc)",
"Growing spices, aromatic, drug and pharmaceutical crops",
"Seeds and plant propagation",
"Raising of cattle",
"Raising of sheep", 
"Raising of swine/pigs",
"Raising of poultry",
"Raising of animals n.e.c.; services to agriculture",
"Forestry and logging",
"Fishing",
"Crustaceans and molluscs",
"Hard coal",
"Lignite and peat",
"Petroleum extraction",
"Gas extraction",
"Iron ores",
"Uranium ores",
"Aluminium ore",
"Copper ores", 
"Gold ores",
"Lead/zinc/silver ores",
"Nickel ores",
"Tin ores",
"Other non-ferrous ores",
"Quarrying of stone, sand and clay",
"Chemical and fertilizer minerals",
"Extraction of salt",
"Mining and quarrying n.e.c.; services to mining",
"Beef meat",
"Sheep meat",
"Pork",
"Poultry meat",
"Other meat products",
"Fish products",
"Cereal products",
"Vegetable products",
"Fruit products",
"Food products and feeds n.e.c.",
"Sugar refining; cocoa, chocolate and confectionery",
"Animal oils and fats",
"Vegetable oils and fats",
"Dairy products",
"Alcoholic and other  beverages",
"Tobacco products",
"Textiles and clothing",
"Leather and footwear",
"Sawmill products",
"Pulp and paper",
"Printing",
"Coke oven products",
"Refined petroleum products",
"Nitrogenous fertilizers",
"Non-nitrogenous and mixed fertilizers",
"Basic petrochemical products",
"Basic inorganic chemicals",
"Basic organic chemicals",
"Pharmaceuticals and medicinal products",
"Dyes, paints, glues, detergents and other chemical products",
"Rubber products",
"Plastic products",
"Clay building materials",
"Other ceramics n.e.c.",
"Cement, lime and plaster products",
"Other non-metallic mineral products n.e.c.",
"Basic iron and steel",
"Basic aluminium",
"Basic Copper",
"Basic Gold",
"Basic lead/zinc/silver",
"Basic nickel",
"Basic tin",
"Basic non-ferrous metals n.e.c.",
"Fabricated metal products",
"Machinery and equipment",
"Motor vehicles, trailers and semi-trailers",
"Other transport equipment",
"Repair and installation of machinery and equipment",
"Computers; electronic products; optical and precision instruments",
"Electrical equipment",
"Furniture and other manufacturing n.e.c",
"Electric power generation, transmission and distribution",
"Distribution of gaseous fuels through mains",
"Water collection, treatment and supply; sewerage",
"Waste collection, treatment, and disposal",
"Materials recovery",
"Building construction",
"Civil engineering construction",
"Wholesale and retail trade; repair of motor vehicles and motorcycles",
"Road transport",
"Rail transport",
"Transport via pipeline",
"Water transport", 
"Air transport",
"Services to transport",
"Postal and courier services",
"Hospitality",
"Publishing",
"Telecommunications",
"Information services",
"Finance and insurance",
"Property and real estate",
"Professional, scientific and technical services",
"Administrative services",
"Government; social security; defence; public order",
"Education",
"Human health and social work activities",
"Arts, entertainment and recreation",
"Other services"]
_regions = ["Rest of Americas",
"Rest of Europe",
"Rest of Africa",
"Rest of Asia-Pacific",
"Afghanistan",
"Angola",
"Albania",
"United Arab Emirates",
"Argentina",
"Armenia",
"Australia",
"Austria",
"Azerbaijan",
"Burundi",
"Belgium",
"Benin",
"Burkina Faso",
"Bangladesh",
"Bulgaria",
"Bahrain",
"Bahamas",
"Bosnia and Herzegovina",
"Belarus",
"Belize",
"Bolivia",
"Brazil",
"Brunei Darussalam",
"Bhutan",
"Botswana",
"Central African Republic",
"Canada",
"Switzerland",
"Chile",
"China",
"Cote d'Ivoire",
"Cameroon",
"DR Congo",
"Rep Congo",
"Colombia",
"Costa Rica",
"Cuba",
"Cyprus",
"CSSR/Czech Republic (1990/1991)",
"Germany",
"Djibouti",
"DR Yemen (Aden)",
"Denmark",
"Dominican Republic",
"Algeria",
"Ecuador",
"Egypt",
"Eritrea",
"Spain",
"Estonia",
"Ethiopia/DR Ethiopia (1992/1993)",
"Finland",
"France",
"Gabon",
"United Kingdom",
"Georgia",
"Ghana",
"Guinea",
"Gambia",
"Equatorial Guinea",
"Greece",
"Guatemala",
"Honduras",
"Hong Kong",
"Croatia",
"Haiti",
"Hungary",
"Indonesia",
"India",
"Ireland",
"Iran",
"Iraq",
"Iceland",
"Israel",
"Italy",
"Jamaica",
"Jordan",
"Japan",
"Kazakhstan",
"Kenya",
"Kyrgyzstan",
"Cambodia",
"South Korea",
"Kuwait",
"Laos",
"Lebanon",
"Liberia",
"Libya",
"Sri Lanka",
"Lithuania",
"Luxembourg",
"Latvia",
"Morocco",
"Moldova",
"Madagascar",
"Mexico",
"Macedonia",
"Mali",
"Malta",
"Myanmar",
"Mongolia",
"Mozambique",
"Mauritania",
"Malawi",
"Malaysia",
"Namibia",
"Niger",
"Nigeria",
"Nicaragua",
"Netherlands",
"Norway",
"Nepal",
"New Zealand",
"Oman",
"Pakistan",
"Palestine",
"Panama",
"Peru",
"Philippines",
"Papua New Guinea",
"Poland",
"North Korea",
"Portugal",
"Paraguay",
"Qatar",
"Romania",
"USSR/Russian Federation (1990/1991)",
"Rwanda",
"Saudi Arabia",
"South Sudan",
"Senegal",
"Singapore",
"Sierra Leone",
"El Salvador",
"Somalia",
"Yugoslavia/Serbia (1991/1992)",
"Sudan/North Sudan (2010/2011)",
"Slovakia",
"Slovenia",
"Sweden",
"Syria",
"Chad",
"Togo",
"Thailand",
"Tajikistan",
"Turkmenistan",
"Tunisia",
"Turkey",
"Tanzania",
"Uganda",
"Ukraine",
"Uruguay",
"United States of America",
"Uzbekistan",
"Venezuela",
"Viet Nam",
"Yemen Arab Republic/Yemen (1990/1991)",
"South Africa",
"Zambia",
"Zimbabwe"]
_Z_multiindex = pd.MultiIndex.from_product(
    [_regions, _sectors], names=["region", "sector"]
)

After we have specified the regions, we need to import the Z matrix. Note, we need to split use tables for BoARIO method (not MRIA). To do this we: 

For Z, the use matrices (demand between sectors in one region) is: 
| R1 | R2 |
|---------|---------|
| 0   | supply   |
| use   | 0   |

Where the trade matrices are everywhere else. For disaster analysis, we need both use and trade matrices. 

For Y, we need to skip every 120 rows as they correlate to the supply matrices. Supply matrices have no final demand, so there are only 0s in these matrices and thus need to be removed. The structure is: 

| Y1 | Y2 |
|---------|---------|
| R1Y1   | R1Y2   |
| 0   | 0   |
| R2Y1   | R2Y2   |
| 0   | 0   |

Where each sector and regions Y, needs to be summed to get the total final demand. 

In [5]:
import numpy as np
import pandas as pd
import csv

# # Read CSV into a NumPy array
# Z_data = pd.read_csv('/Users/cmor7802/repos/climpyplots/boario/data/tmp/gloria59/20240111_120secMother_AllCountries_002_T-Results_2009_059_Markup001(full)(1).csv', header=None).values

# block_size = 120
# total_rows, total_cols = Z_data.shape

# def calculate_indices(total_size, block_size, include_first):
#     indices = []
#     start = 0 if include_first else block_size
#     for i in range(start, total_size, 2 * block_size):
#         end = min(i + block_size, total_size)
#         indices.extend(range(i, end))
#     return indices

# row_indices = calculate_indices(total_rows, block_size, include_first=False)
# col_indices = calculate_indices(total_cols, block_size, include_first=True)

# # Slice the data per the indicies
# new_Z_matrix = Z_data[np.ix_(row_indices, col_indices)]

# # Save as CSV
# np.savetxt('Zsliced_2009.csv', new_Z_matrix, delimiter=",", fmt='%s')

# Optionally, save as pickle for fast loading later
# import pickle
# with open('Zsliced_2009.pkl', 'wb') as f:
#     pickle.dump(new_Z_matrix, f)

# print("New Matrix Dimensions:", new_Z_matrix.shape)

new_Z_matrix = pd.read_csv('/Users/cmor7802/repos/climpyplots/boario/data/tmp/gloria59/Zsliced_2009.csv', header=None).values

In [6]:
Z = pd.DataFrame(
    data=new_Z_matrix, index=_Z_multiindex, columns=_Z_multiindex
)
Z

region                                                              Rest of Americas  \
sector                                                                 Growing wheat   
region           sector                                                                
Rest of Americas Growing wheat                                             27.120000   
                 Growing maize                                              0.000001   
                 Growing cereals n.e.c                                      0.000003   
                 Growing leguminous crops and oil seeds                     0.000004   
                 Growing rice                                               0.000002   
...                                                                              ...   
Zimbabwe         Government; social security; defence; public order         0.000005   
                 Education                                                  0.000005   
                 Human health and social work activities                    0.000004   
                 Arts, entertainment and recreation                         0.000006   
                 Other services                                             0.000008   

region                                                                             \
sector                                                              Growing maize   
region           sector                                                             
Rest of Americas Growing wheat                                           0.000003   
                 Growing maize                                           0.002953   
                 Growing cereals n.e.c                                   0.000006   
                 Growing leguminous crops and oil seeds                  0.000009   
                 Growing rice                                            0.000004   
...                                                                           ...   
Zimbabwe         Government; social security; defence; public order      0.000007   
                 Education                                               0.000007   
                 Human health and social work activities                 0.000006   
                 Arts, entertainment and recreation                      0.000009   
                 Other services                                          0.000012   

region                                                                                     \
sector                                                              Growing cereals n.e.c   
region           sector                                                                     
Rest of Americas Growing wheat                                                   0.000002   
                 Growing maize                                                   0.000002   
                 Growing cereals n.e.c                                        2575.800000   
                 Growing leguminous crops and oil seeds                          0.000007   
                 Growing rice                                                    0.000003   
...                                                                                   ...   
Zimbabwe         Government; social security; defence; public order              0.000004   
                 Education                                                       0.000005   
                 Human health and social work activities                         0.000004   
                 Arts, entertainment and recreation                              0.000006   
                 Other services                                                  0.000008   

region                                                                                                      \
sector                                                              Growing leguminous crops and oil seeds   
region           sector                              

We now do a similar thing for final demand. As with before, we need to separate the use tables from the supply tables. TBC. 

In [7]:
_categories = ["Household final consumption",
               "Non-profit institutions serving households",
               "Government final consumption",
               "Gross fixed capital formation",
               "Changes in inventories",
               "Acquisitions less disposals of valuables"]
_fd_multiindex = pd.MultiIndex.from_product(
    [_regions, _categories], names=["region", "category"]
)

In [8]:
import numpy as np
import pandas as pd
import csv

# # Read CSV into a NumPy array -- this is GLORIA 059!
# Y_data = pd.read_csv('/Users/cmor7802/repos/climpyplots/boario/data/tmp/gloria57/20240111_120secMother_AllCountries_002_Y-Results_2009_059_Markup001(full)(1).csv', header=None).values

# block_size = 120
# total_rows, total_cols = Y_data.shape

# def calculate_indices(total_size, block_size, include_first):
#     indices = []
#     start = 0 if include_first else block_size
#     for i in range(start, total_size, 2 * block_size):
#         end = min(i + block_size, total_size)
#         indices.extend(range(i, end))
#     return indices

# row_indices = calculate_indices(total_rows, block_size, include_first=False)
# col_indices = list(range(total_cols)) # we are not skipping any columns, just the rows from the supply tables

# # Slice the data per the indicies
# new_Y_matrix = Y_data[np.ix_(row_indices, col_indices)]

# # Save as CSV
# np.savetxt('Ysliced_2009.csv', new_Y_matrix, delimiter=",", fmt='%s')

# # Optionally, save as pickle for fast loading later
# # import pickle
# # with open('Ysliced_2009.pkl', 'wb') as f:
# #     pickle.dump(new_Y_matrix, f)

# print("New Matrix Dimensions:", new_Y_matrix.shape)

new_Y_matrix = pd.read_csv('/Users/cmor7802/repos/climpyplots/boario/data/tmp/gloria59/Ysliced_2009.csv', header=None).values

In [9]:
# # Some debugging code
# print("Y_data shape:", Y_data.shape)
# print("Row indices:", row_indices[:10], "...", row_indices[-10:])
# print("Col indices:", col_indices[:10], "...", col_indices[-10:])
# print("Expected shape:", len(row_indices), len(col_indices))

In [10]:
Y = pd.DataFrame(
    data=new_Y_matrix, index=_Z_multiindex, columns=_fd_multiindex
)
Y

# we can tell that this looks correct because the supply matrices would have 0s 

region                                                                         Rest of Americas  \
category                                                            Household final consumption   
region           sector                                                                           
Rest of Americas Growing wheat                                                     2.435200e+02   
                 Growing maize                                                     5.030000e-02   
                 Growing cereals n.e.c                                             1.685300e+02   
                 Growing leguminous crops and oil seeds                            5.328600e+02   
                 Growing rice                                                      1.142700e+05   
...                                                                                         ...   
Zimbabwe         Government; social security; defence; public order                6.643900e-07   
                 Education                                                         4.866600e-07   
                 Human health and social work activities                           4.817400e-07   
                 Arts, entertainment and recreation                                6.161500e-07   
                 Other services                                                    6.180300e-07   

region                                                                                                          \
category                                                            Non-profit institutions serving households   
region           sector                                                                                          
Rest of Americas Growing wheat                                                                        0.000022   
                 Growing maize                                                                        0.000006   
                 Growing cereals n.e.c                                                                0.000029   
                 Growing leguminous crops and oil seeds                                               0.000039   
                 Growing rice                                                                         0.000017   
...                                                                                                        ...   
Zimbabwe         Government; social security; defence; public order                                   0.000003   
                 Education                                                                            0.000002   
                 Human health and social work activities                                              0.000002   
                 Arts, entertainment and recreation                                                   0.000002   
                 Other services                                                                       0.000004   

region                                                                                            \
category                                                            Government final consumption   
region           sector                                                                            
Rest of Americas Growing wheat                                                          0.002190   
                 Growing maize                                                          0.000643   
                 Growing cereals n.e.c                                                  0.002902   
                 Growing leguminous crops and oil seeds                                 0.003983   
                 Growing rice                                                           0.001759   
...                                                                                          ...   
Zimbabwe         Government; social security; defence; public order                     0.000001   
                

In [11]:
_va_categories = ["Compensation of employees",
                  "Taxes on production",
                  "Subsidies on production",
                  "Net operating surplus",
                  "Net mixed income",
                  "Consumption of fixed capital"]
_va_multiindex = pd.MultiIndex.from_product(
    [_regions, _va_categories], names=["region", "category"]
)

In [12]:
import numpy as np
import pandas as pd
import csv

# # Read CSV into a NumPy array -- this is GLORIA 059!
# VA_data = pd.read_csv('/Users/cmor7802/repos/climpyplots/boario/data/tmp/gloria57/20240419_120secMother_AllCountries_002_V-Results_2009_059_Markup001(full)(1).csv', header=None).values

# block_size = 120
# total_rows, total_cols = VA_data.shape

# def calculate_indices(total_size, block_size, include_first):
#     indices = []
#     start = 0 if include_first else block_size
#     for i in range(start, total_size, 2 * block_size):
#         end = min(i + block_size, total_size)
#         indices.extend(range(i, end))
#     return indices

# col_indices = calculate_indices(total_cols, block_size, include_first=False)
# row_indices = list(range(total_rows)) # we are not skipping any rows, just the columns from the supply tables

# # Slice the data per the indicies
# new_VA_matrix = VA_data[np.ix_(row_indices, col_indices)]

# # Save as CSV
# np.savetxt('VAsliced_2009.csv', new_VA_matrix, delimiter=",", fmt='%s')

# # Optionally, save as pickle for fast loading later
# # import pickle
# # with open('VAsliced_2009.pkl', 'wb') as f:
# #     pickle.dump(new_VA_matrix, f)

# print("New Matrix Dimensions:", new_VA_matrix.shape)

new_VA_matrix = pd.read_csv('/Users/cmor7802/repos/climpyplots/boario/data/tmp/gloria59/VAsliced_2009.csv', header=None).values

In [13]:
VA = pd.DataFrame(
    data=new_VA_matrix, index=_va_multiindex, columns=_Z_multiindex
)
VA

region                                        Rest of Americas                \
sector                                           Growing wheat Growing maize   
region           category                                                      
Rest of Americas Compensation of employees                 0.0           0.0   
                 Taxes on production                       0.0           0.0   
                 Subsidies on production                   0.0           0.0   
                 Net operating surplus                     0.0           0.0   
                 Net mixed income                          0.0           0.0   
...                                                        ...           ...   
Zimbabwe         Taxes on production                       0.0           0.0   
                 Subsidies on production                   0.0           0.0   
                 Net operating surplus                     0.0           0.0   
                 Net mixed income                          0.0           0.0   
                 Consumption of fixed capital              0.0           0.0   

region                                                               \
sector                                        Growing cereals n.e.c   
region           category                                             
Rest of Americas Compensation of employees                      0.0   
                 Taxes on production                            0.0   
                 Subsidies on production                        0.0   
                 Net operating surplus                          0.0   
                 Net mixed income                               0.0   
...                                                             ...   
Zimbabwe         Taxes on production                            0.0   
                 Subsidies on production                        0.0   
                 Net operating surplus                          0.0   
                 Net mixed income                               0.0   
                 Consumption of fixed capital                   0.0   

region                                                                                \
sector                                        Growing leguminous crops and oil seeds   
region           category                                                              
Rest of Americas Compensation of employees                                       0.0   
                 Taxes on production                                             0.0   
                 Subsidies on production                                         0.0   
                 Net operating surplus                                           0.0   
                 Net mixed income                                                0.0   
...                                                                              ...   
Zimbabwe         Taxes on production                                             0.0   
                 Subsidies on production                                         0.0   
                 Net operating surplus                                           0.0   
                 Net mixed income                                                0.0   
                 Consumption of fixed capital                                    0.0   

region                                                      \
sector                                        Growing rice   
region           category                                    
Rest of Americas Compensation of employees             0.0   
                 Taxes on production                   0.0   
                 Subsidies on production               0.0   
                 Net operating surplus                 0.0   
                 Net mixed income                      0.0   
...                                                    ...   
Zimbabwe         Taxes on production                   0.0   
                 Subsidies on p

In [14]:
# col_indices = list(range(total_cols))
# new_VA_matrix = VA_data[:, col_indices]
# print("New Matrix Dimensions:", new_VA_matrix.shape)
# print(new_VA_matrix[:10, :10])  # Check for non-zero values

In [15]:
# # Find all non-zero entries
# nonzero_indices = np.argwhere(VA_data != 0)
# print("First 10 non-zero entries (row, col):", nonzero_indices[:10])

# print("Row sums (first 20):", VA_data.sum(axis=1)[:20])
# print("Column sums (first 20):", VA_data.sum(axis=0)[:20])

# row = 700 # this should be a va category for russia index (131*2)
# column = 15601 # this should be Russian wheat
# print("Row {}:".format(row), VA_data[row, column])

Now we want to initiate the data as an IOSystem. We do this following the pymrio guidelines. https://pymrio.readthedocs.io/en/latest/notebooks/pymrio_directly_assign_attributes.html

In [16]:
import pymrio
gloria_io = pymrio.IOSystem()

gloria_io.Z = Z
gloria_io.Y = Y

factor_input = pymrio.Extension(name="Value Added", F=VA)
gloria_io.VA = factor_input

str(gloria_io)

'IO System with parameters: Z, Y, meta, VA'

We need to calculate x and A for our analysis. x = total output vector (column vector) and A = technical coefficients matrix. 

In [17]:
import pymrio

# For x (total output vector)
x = pymrio.calc_x(Z, Y)  # Z and Y are your matrices
gloria_io.x = x

# For A (technical coefficients matrix)
A = pymrio.calc_A(Z, x)
gloria_io.A = A

x_df = pd.DataFrame(data=x, index=_Z_multiindex, columns=['indout']) # the column for the output vector is not 'x' it is 'indout'. We found this out by printing the x_df.columns
print(x_df.head())

str(gloria_io)


                                                                indout
region           sector                                               
Rest of Americas Growing wheat                           158032.460796
                 Growing maize                           205079.474825
                 Growing cereals n.e.c                    42935.722924
                 Growing leguminous crops and oil seeds   24501.586802
                 Growing rice                            179761.904888


'IO System with parameters: Z, Y, x, A, meta, VA'

Now we need to save the data in our IOSystem to pkled files for the calculations. As we are using GLORIA, which does not have a pymrio.parse, we need to simply create the pickle directory and then save the gloria_io files to our new directory. 

In [18]:
import pymrio
import os

gloria_storage = "data/tmp/gloria59"

# Create the directory
gloria_pkl = "data/tmp/gloria/Gloria59_2009_pkl/"
os.makedirs(gloria_pkl, exist_ok=True)

## Pickle
# Save to pickle format
gloria_io.save_all(path=gloria_pkl, table_format="pkl")

# Load the pickled data
gloria_io = pymrio.load_all(path=gloria_pkl)

## Setting up the model

Now we can set up the model and the event class to run the simulation. 

In [19]:
import pymrio
import warnings
# import the different classes
from boario.extended_models import ARIOPsiModel  # The core of the model

# Suppress the FutureWarning if you want
warnings.simplefilter(action="ignore", category=FutureWarning)

# 1. Load the Eora data
gloria = pymrio.load_all(path=gloria_pkl)

model = ARIOPsiModel(
    pym_mrio=gloria,
    order_type="alt",
    alpha_base=1.0,
    alpha_max=1.25,
    alpha_tau=365,
    rebuild_tau=60,
    main_inv_dur=90,
    monetary_factor=10**6,
    temporal_units_by_step=1,
    iotable_year_to_temporal_unit_factor=365
)

/opt/anaconda3/envs/scmods/lib/python3.12/site-packages/boario/model_base.py:252: UserWarning: Found negative values in the value added, will set to 0. Note that industries with null value added will have a null productive capital if it is defined from value added.
                industries with negative VA: MultiIndex([('Afghanistan', ...),
            ('Afghanistan', ...),
            ('Afghanistan', ...),
            ('Afghanistan', ...),
            ('Afghanistan', ...),
            ('Afghanistan', ...),
            ('Afghanistan', ...),
            (    'Albania', ...),
            (    'Algeria', ...),
            (     'Angola', ...),
            ...
            (     'Zambia', ...),
            (   'Zimbabwe', ...),
            (   'Zimbabwe', ...),
            (   'Zimbabwe', ...),
            (   'Zimbabwe', ...),
            (   'Zimbabwe', ...),
            (   'Zimbabwe', ...),
            (   'Zimbabwe', ...),
            (   'Zimbabwe', ...),
            (   'Zimbabwe',

There are three event types in BoARIO. Likely the most relevant for an agricultural shock is the third type: **EventArbitProd**. 

According to BoARIO documentation: When creating this type of event, the impact values should be value between 0 and 1 stating the fraction of production capacity unavailable due to the event.

As for EventKapitalRecover, a recovery function and a recovery time may be given. Otherwise, production capacity is restored instantaneously after the duration of the event has elapsed.

Source: https://spjuhel.github.io/BoARIO/tutorials/boario-events.html

In [20]:
from boario import event

import pandas as pd
heatwave_10 = pd.Series(
    data=[0.1],
    index=pd.MultiIndex.from_product(
        [["USSR/Russian Federation (1990/1991)"], ["Growing wheat"]], names=["region", "sector"]
    ),
)
heatwave_10

ev = event.from_series(
    impact=heatwave_10,
    event_type="arbitrary",
    occurrence=1, #occurrence, state 1 for one event?
    duration=14, # duration of the event
    recovery_function="linear",
    recovery_tau=5,
)

Now we can initiate the simulation. 

In [22]:
from boario.simulation import Simulation

# Create a new simulation with your Eora model
sim = Simulation(
    model=model,  # This is your ARIOPsiModel with Eora data
    n_temporal_units_to_sim=15,  # simulation length
    register_stocks=True,  # track stocks
    show_progress=True,  # show progress during simulation
    save_events=True, 
    save_params=True, 
    save_index=True, 
    save_records=[], 
    boario_output_dir='/Users/cmor7802/repos/climpyplots/boario/results/boario-gloria-09', 
    results_dir_name=None, 
)

# Now try adding the event
sim.add_event(ev)

# Launch the simulation
sim.loop()

Processed: Step: 0 ~   0% ETA:  --:--:--


: 

In [ ]:
from boario.simulation import Simulation

# Create a new simulation with your Eora model
sim = Simulation(
    model=model,  # This is your ARIOPsiModel with Eora data
    n_temporal_units_to_sim=15,  # simulation length
    register_stocks=True,  # track stocks
    show_progress=True,  # show progress during simulation
    save_events=True, 
    save_params=True, 
    save_index=True, 
    save_records=[], 
    boario_output_dir='/Users/cmor7802/repos/climpyplots/boario/results/boario-gloria-09', 
    results_dir_name=None, 
)

# Now try adding the event
sim.add_event(ev)

# Launch the simulation
sim.loop()

Processed: Step: 0 ~   0% ETA:  --:--:--


: 